In [1]:
import pandas as pd
import numpy as np
from dateutil import parser
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

In [3]:
#Loading the csv file
df = pd.read_csv("data/jiji_housing_raw.csv")

df.head()

,title,property_size,bedrooms,bathrooms,furnishing,region,region_name,region_parent_name,is_boost,price
0,Furnished 3bdrm Apartment in Maitama for sale,500,3,4,furnished,"Abuja (FCT), Maitama",Maitama,Abuja (FCT),diamond,"₦ 580,000,000"
1,4bdrm Townhouse/Terrace in Gaduwa for sale,380,4,4,unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),premium,"₦ 320,000,000"
2,4bdrm Townhouse/Terrace in Gaduwa for sale,300,4,4,unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),premium,"₦ 150,000,000"
3,4bdrm House in Off Lekki-Epe Expressway for sale,500,4,4,unfurnished,"Ajah, Off Lekki-Epe Expressway",Off Lekki-Epe Expressway,Ajah,premium,"₦ 165,000,000"
4,5bdrm Duplex in Lekki for sale,350,5,5,unfurnished,"Lagos State, Lekki",Lekki,Lagos State,premium,"₦ 550,000,000"


In [4]:
#Missing value check

df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   title               2000 non-null   str  
 1   property_size       2000 non-null   int64
 2   bedrooms            2000 non-null   int64
 3   bathrooms           2000 non-null   int64
 4   furnishing          2000 non-null   str  
 5   region              2000 non-null   str  
 6   region_name         2000 non-null   str  
 7   region_parent_name  1996 non-null   str  
 8   is_boost            2000 non-null   str  
 9   price               2000 non-null   str  
dtypes: int64(3), str(7)
memory usage: 379.7 KB


In [5]:
df.columns = df.columns.str.title()
df.columns


Index(['Title', 'Property_Size', 'Bedrooms', 'Bathrooms', 'Furnishing',
       'Region', 'Region_Name', 'Region_Parent_Name', 'Is_Boost', 'Price'],
      dtype='str')

In [6]:
#Clean price

df['Price'] = (
    df['Price']
    .astype(str)
    .replace(r"[₦,]", "", regex=True)
    .astype(float)
)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               2000 non-null   str    
 1   Property_Size       2000 non-null   int64  
 2   Bedrooms            2000 non-null   int64  
 3   Bathrooms           2000 non-null   int64  
 4   Furnishing          2000 non-null   str    
 5   Region              2000 non-null   str    
 6   Region_Name         2000 non-null   str    
 7   Region_Parent_Name  1996 non-null   str    
 8   Is_Boost            2000 non-null   str    
 9   Price               2000 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 350.3 KB


In [7]:
#Dealing with Outliers

Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)

IQR = Q3 - Q1

#Finding the Lower and upper bounds
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

outliers = (df['Price'] < lower_bound) | (df['Price'] > upper_bound)

print('Outliers:', outliers.sum())

Outliers: 196


In [8]:
#Excluding Outliers from dataframe using ""

lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

new_df = df[
    (df['Price'] > lower_bound) & (df['Price'] < upper_bound)
]

new_df.info()

<class 'pandas.DataFrame'>
Index: 1804 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1804 non-null   str    
 1   Property_Size       1804 non-null   int64  
 2   Bedrooms            1804 non-null   int64  
 3   Bathrooms           1804 non-null   int64  
 4   Furnishing          1804 non-null   str    
 5   Region              1804 non-null   str    
 6   Region_Name         1804 non-null   str    
 7   Region_Parent_Name  1800 non-null   str    
 8   Is_Boost            1804 non-null   str    
 9   Price               1804 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 331.1 KB


In [9]:
#Convert furnishing type to consistent categories

new_df['Furnishing'] = new_df['Furnishing'].astype(str).str.title()

new_df.head()

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,Furnished 3bdrm Apartment in Maitama for sale,500,3,4,Furnished,"Abuja (FCT), Maitama",Maitama,Abuja (FCT),diamond,580000000.00
1,4bdrm Townhouse/Terrace in Gaduwa for sale,380,4,4,Unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),premium,320000000.00
2,4bdrm Townhouse/Terrace in Gaduwa for sale,300,4,4,Unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),premium,150000000.00
3,4bdrm House in Off Lekki-Epe Expressway for sale,500,4,4,Unfurnished,"Ajah, Off Lekki-Epe Expressway",Off Lekki-Epe Expressway,Ajah,premium,165000000.00
4,5bdrm Duplex in Lekki for sale,350,5,5,Unfurnished,"Lagos State, Lekki",Lekki,Lagos State,premium,550000000.00


In [10]:

#Convert is_boost type to consistent categories

new_df['Is_Boost'] = new_df['Is_Boost'].astype(str).str.title()

new_df.head()


,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,Furnished 3bdrm Apartment in Maitama for sale,500,3,4,Furnished,"Abuja (FCT), Maitama",Maitama,Abuja (FCT),Diamond,580000000.00
1,4bdrm Townhouse/Terrace in Gaduwa for sale,380,4,4,Unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),Premium,320000000.00
2,4bdrm Townhouse/Terrace in Gaduwa for sale,300,4,4,Unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),Premium,150000000.00
3,4bdrm House in Off Lekki-Epe Expressway for sale,500,4,4,Unfurnished,"Ajah, Off Lekki-Epe Expressway",Off Lekki-Epe Expressway,Ajah,Premium,165000000.00
4,5bdrm Duplex in Lekki for sale,350,5,5,Unfurnished,"Lagos State, Lekki",Lekki,Lagos State,Premium,550000000.00


In [11]:
new_df['Is_Boost'] = new_df['Is_Boost'].replace('Vip_Gold', 'VIP Gold')


In [12]:
new_df['Is_Boost'] = new_df['Is_Boost'].replace('Vip', 'VIP')
new_df.head()

,Title,Property_Size,Bedrooms,Bathrooms,Furnishing,Region,Region_Name,Region_Parent_Name,Is_Boost,Price
0,Furnished 3bdrm Apartment in Maitama for sale,500,3,4,Furnished,"Abuja (FCT), Maitama",Maitama,Abuja (FCT),Diamond,580000000.00
1,4bdrm Townhouse/Terrace in Gaduwa for sale,380,4,4,Unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),Premium,320000000.00
2,4bdrm Townhouse/Terrace in Gaduwa for sale,300,4,4,Unfurnished,"Abuja (FCT), Gaduwa",Gaduwa,Abuja (FCT),Premium,150000000.00
3,4bdrm House in Off Lekki-Epe Expressway for sale,500,4,4,Unfurnished,"Ajah, Off Lekki-Epe Expressway",Off Lekki-Epe Expressway,Ajah,Premium,165000000.00
4,5bdrm Duplex in Lekki for sale,350,5,5,Unfurnished,"Lagos State, Lekki",Lekki,Lagos State,Premium,550000000.00


In [13]:
total_duplicates = new_df.duplicated().sum()
print(total_duplicates)

47


In [14]:
new_df = new_df.drop_duplicates()
new_df.duplicated().sum()

np.int64(0)

In [15]:

new_df['Region_Parent_Name'] = new_df['Region_Parent_Name'].replace(['Lekki', 'Ikorodu', 'Ajah', 'Ojodu', 'Ikeja', 'Ipaja', 'Egbeda', 'Agege', 'Ibeju', 
                                      'Magodo', 'Gbagada', 'Shomolu', 'Ojo', 'Lagos Island (Eko)', 'Ogudu', 'Isolo', 'Kosofe', 
                                      'Ikotun/Igando', 'Egbe/Idimu', 'Yaba', 'Ifako-Ijaiye', 'Alimosho', 'Ikoyi', 'Badagry', 
                                      'Apapa', 'Surulere', 'Oshodi', 'Ogba', 'Ilupeju', 'Ojota', 'Epe', 'Mushin', 'Amuwo-Odofin', 'Ejigbo', 'Maryland', 'Victoria Island'], 'Lagos State')

new_df['Region_Parent_Name'].unique()

<ArrowStringArray>
[      'Abuja (FCT)',       'Lagos State',       'Delta State',
 'Cross River State',       'Enugu State',    'Lugbe District',
            'Ibadan',         'Imo State',              'Jiwa',
        'Ogun State',              'Wuse',                 nan,
           'Katampe',          'Gwarinpa',    'Nasarawa State',
           'Garki 1',         'Edo State',        'Osun State',
         'Oyo State',      'Rivers State',             'Bwari',
   'Akwa Ibom State',     'Port-Harcourt',       'Kwara State',
      'Apo District',      'Kaduna State',            'Gwagwa',
        'Ondo State',      'Jigawa State',            'Sagamu',
        'Abia State']
Length: 31, dtype: str

In [16]:
new_df['Region_Parent_Name'] = new_df['Region_Parent_Name'].replace(['Apo District', 'Lugbe District', 'Bwari', 'Gwarinpa', 'Gwagwa', 'Katampe', 'Jiwa', 'Garki 1', 'Wuse'], 'Abuja (FCT)')

new_df['Region_Parent_Name'].unique()

<ArrowStringArray>
[      'Abuja (FCT)',       'Lagos State',       'Delta State',
 'Cross River State',       'Enugu State',            'Ibadan',
         'Imo State',        'Ogun State',                 nan,
    'Nasarawa State',         'Edo State',        'Osun State',
         'Oyo State',      'Rivers State',   'Akwa Ibom State',
     'Port-Harcourt',       'Kwara State',      'Kaduna State',
        'Ondo State',      'Jigawa State',            'Sagamu',
        'Abia State']
Length: 22, dtype: str

In [17]:
new_df['Region_Parent_Name'] = new_df['Region_Parent_Name'].replace('Ibadan', 'Oyo State')
new_df['Region_Parent_Name'].value_counts()



Region_Parent_Name
Lagos State          998
Abuja (FCT)          478
Oyo State            169
Port-Harcourt         29
Edo State             13
Rivers State          12
Kwara State           12
Ogun State             8
Imo State              7
Enugu State            5
Osun State             5
Delta State            4
Ondo State             4
Nasarawa State         3
Cross River State      1
Akwa Ibom State        1
Kaduna State           1
Jigawa State           1
Sagamu                 1
Abia State             1
Name: count, dtype: int64

In [18]:
new_df['Region_Parent_Name'] = new_df['Region_Parent_Name'].replace('Sagamu', 'Ogun State')
new_df['Region_Parent_Name'].value_counts()


Region_Parent_Name
Lagos State          998
Abuja (FCT)          478
Oyo State            169
Port-Harcourt         29
Edo State             13
Rivers State          12
Kwara State           12
Ogun State             9
Imo State              7
Enugu State            5
Osun State             5
Delta State            4
Ondo State             4
Nasarawa State         3
Cross River State      1
Akwa Ibom State        1
Kaduna State           1
Jigawa State           1
Abia State             1
Name: count, dtype: int64

In [19]:
new_df.info()

<class 'pandas.DataFrame'>
Index: 1757 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1757 non-null   str    
 1   Property_Size       1757 non-null   int64  
 2   Bedrooms            1757 non-null   int64  
 3   Bathrooms           1757 non-null   int64  
 4   Furnishing          1757 non-null   str    
 5   Region              1757 non-null   str    
 6   Region_Name         1757 non-null   str    
 7   Region_Parent_Name  1753 non-null   str    
 8   Is_Boost            1757 non-null   str    
 9   Price               1757 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 328.1 KB


In [20]:
new_df = new_df.dropna()

new_df.info()

<class 'pandas.DataFrame'>
Index: 1753 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1753 non-null   str    
 1   Property_Size       1753 non-null   int64  
 2   Bedrooms            1753 non-null   int64  
 3   Bathrooms           1753 non-null   int64  
 4   Furnishing          1753 non-null   str    
 5   Region              1753 non-null   str    
 6   Region_Name         1753 non-null   str    
 7   Region_Parent_Name  1753 non-null   str    
 8   Is_Boost            1753 non-null   str    
 9   Price               1753 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 327.4 KB


In [21]:

new_df.to_csv('data/jiji_housing_cleaned.csv', index=False)
new_df.info()



<class 'pandas.DataFrame'>
Index: 1753 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Title               1753 non-null   str    
 1   Property_Size       1753 non-null   int64  
 2   Bedrooms            1753 non-null   int64  
 3   Bathrooms           1753 non-null   int64  
 4   Furnishing          1753 non-null   str    
 5   Region              1753 non-null   str    
 6   Region_Name         1753 non-null   str    
 7   Region_Parent_Name  1753 non-null   str    
 8   Is_Boost            1753 non-null   str    
 9   Price               1753 non-null   float64
dtypes: float64(1), int64(3), str(6)
memory usage: 327.4 KB


In [22]:
#Exploratory Data Analysis (EDA)


In [39]:
# What is the average house price in Nigeria?

average_price = round(new_df['Price'].mean(), 2)
print('Housing Average Price(#)', average_price)


Housing Average Price(#) 297935339.42


In [46]:
# Which state has the highest and lowest mean property price?

round(new_df.groupby('Region_Parent_Name')['Price'].mean(), 2).sort_values(ascending=False).head(20).reset_index(name='Average Price')

,Region_Parent_Name,Average Price
0,Jigawa State,870000000.00
1,Lagos State,336748446.89
2,Abuja (FCT),297384937.24
3,Rivers State,254583333.33
4,Enugu State,249000000.00
5,Port-Harcourt,232586206.90
6,Abia State,210000000.00
7,Imo State,172928571.43
8,Oyo State,156710650.89
9,Nasarawa State,151666666.67


In [47]:
# What is the distribution of property sizes?

property_counts = new_df['Property_Size'].value_counts().sort_values(ascending=False).head(20).reset_index(name='Count')
print(property_counts)

    Property_Size  Count
0             500    288
1            1000    159
2             600    155
3             300    151
4             450    110
5             400    110
6             250     86
7             350     75
8             100     69
9             200     49
10            700     37
11            650     30
12            800     30
13            150     29
14            750     23
15            900     21
16            550     19
17            465     11
18           1500     11
19           1200     10


In [44]:
# Which regions have the highest number of listings?

region_counts = new_df['Region_Parent_Name'].value_counts().sort_values(ascending=False).head(20).reset_index(name='Count')
print(region_counts)

   Region_Parent_Name  Count
0         Lagos State    998
1         Abuja (FCT)    478
2           Oyo State    169
3       Port-Harcourt     29
4           Edo State     13
5        Rivers State     12
6         Kwara State     12
7          Ogun State      9
8           Imo State      7
9         Enugu State      5
10         Osun State      5
11        Delta State      4
12         Ondo State      4
13     Nasarawa State      3
14  Cross River State      1
15    Akwa Ibom State      1
16       Kaduna State      1
17       Jigawa State      1
18         Abia State      1


In [27]:
new_df.groupby('Region_Parent_Name')['Price'].mean().round(2).sort_values(ascending=False).reset_index(name='Price')

,Region_Parent_Name,Price
0,Jigawa State,870000000.00
1,Lagos State,336748446.89
2,Abuja (FCT),297384937.24
3,Rivers State,254583333.33
4,Enugu State,249000000.00
5,Port-Harcourt,232586206.90
6,Abia State,210000000.00
7,Imo State,172928571.43
8,Oyo State,156710650.89
9,Nasarawa State,151666666.67


In [48]:
#Which regions dominate premium property sales?

premium_df = new_df[new_df['Is_Boost'] == 'Premium']

region_totals = premium_df['Region_Parent_Name'].value_counts()

print(region_totals)

Region_Parent_Name
Lagos State    16
Abuja (FCT)     6
Oyo State       1
Ogun State      1
Delta State     1
Name: count, dtype: int64


In [29]:
#Are furnished apartments more expensive on average?

furnished = new_df.groupby('Furnishing')['Price'].mean().sort_values(ascending=False).head(20).reset_index(name='Average')
print(furnished)



       Furnishing      Average
0       Furnished 320067987.80
1  Semi-Furnished 317578325.12
2     Unfurnished 274378860.29


In [38]:
#How strongly do bedrooms influence price??

bedrooms = new_df.groupby('Bedrooms')['Price'].mean().sort_values(ascending=False).head(20).reset_index(name='Average')
print(bedrooms)

    Bedrooms      Average
0          9 740000000.00
1          5 480644871.79
2          6 391457971.01
3          7 388500000.00
4          4 294396347.03
5         20 290000000.00
6          2 183488023.95
7          3 181410133.33
8          8 178487500.00
9         10 167600000.00
10        15 165000000.00
11        19 112500000.00
12        12 102780000.00
13        11  97500000.00
14        13  95000000.00
15         1  84790540.54
16        18  80000000.00
17        16  35000000.00


In [37]:
#How strongly do bathrooms influence price??

bathrooms = new_df.groupby('Bathrooms')['Price'].mean().sort_values(ascending=False).head(20).reset_index(name='Average')
print(bathrooms)

    Bathrooms      Average
0           9 516666666.67
1           7 425500000.00
2           5 413531395.35
3           6 402564864.86
4           1 321844059.41
5           8 266112500.00
6           4 257162199.31
7          13 255000000.00
8          20 232500000.00
9          10 215000000.00
10         14 210000000.00
11          3 195127675.28
12          2 182552083.33
13         15 165000000.00
14         12 129750000.00
15         18  80000000.00
16         19  60000000.00
17         11  55000000.00


In [ ]:
#How strongly does property size influence price??

propsize = new_df.groupby('Property_Size')['Price'].mean().sort_values(ascending=False).head(20).reset_index(name='Average')
print(propsize)

In [52]:
#Do boosted/enterprise listings have higher prices?

boosted = new_df.groupby('Is_Boost')['Price'].mean().sort_values(ascending=False).head(20).reset_index(name='Mean Price')
print(boosted)

     Is_Boost   Mean Price
0  Enterprise 330426882.35
1     Diamond 280767785.23
2         VIP 273593750.00
3    VIP Gold 202666666.67
4     Premium 183320000.00
5       False 180528888.89
6       Basic  89600000.00


Summary of Findings

Our analysis shows that Lagos State has the highest number of listings (998) and is the second-highest-priced region, only behind Jigawa, which listed only one house. Furnishing also affects pricing because furnished apartments have the highest average prices (N320,067,987.80) compared to unfurnished (N274,378,860.29). Bedroom numbers, property sizes, and bathroom number doesn't strongly affect listing prices. Enterprise houses are more expensive, averaging (N330,426,882.35), and Lagos has more Luxury buildings listed. 